# JPQ Replicability: MSMARCO TREC Deep Learning track

This notebook downloads everything needed to in order to reproduce the TCT JPQ performance shown in Table 2 of our SIGIR 2026 replicability paper.

```bibtex
@inproceedings{10.1145/3805712.3808565,
author = {Macdonald, Craig and Shen, Zhlli and Tonellotto, Nicola},
title = {A Replicability Study of Joint Product Quantisation for Effective Space-Efficient Dense Retrieval},
year = {2026},
url = {https://doi.org/10.1145/3805712.3808565},
doi = {10.1145/3805712.3808565},
booktitle = {Proceedings of the 49th International ACM SIGIR Conference on Research and Development in Information Retrieval},
numpages = {12},
series = {SIGIR '26}
}
```

You need only 1.3GB of space for model and notebook.

First, we'll install the [pyterrier_dr](https://github.com/terrierteam/pyterrier_dr) repo, which includes all the JPQ code from our SIGIR reproducibility track paper.

In [1]:
%pip install -q 'pyterrier_dr[jpq]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.9/222.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 10.5 MB/s eta 0:00:00


We also install FAISS, and ensure the new transformers 5 isnt used.

In [2]:
%pip install -q faiss-cpu "transformers==4.57.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import transformers
transformers.__version__

'4.57.1'

In [4]:
import pyterrier_dr
import pyterrier_dr.jpq
import pyterrier as pt

We'll download a JPQ TCT index and model checkpoint. These are contained within the following HuggingFace dataset:

> https://huggingface.co/datasets/jpq-repro/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__


In [5]:
index = pyterrier_dr.jpq.JPQIndex.from_hf("jpq-repro/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__")

https://huggingface.co/datasets/jpq-repro/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744_…

extracting codes.f4 [809.5 MB]
extracting config.json [612 B]
extracting docnos.npids [206 B]
extracting file_inventory.csv [180 B]
extracting model.safetensors [417.7 MB]
extracting opq.f4 [2.2 MB]
extracting pt_meta.json [106 B]
extracting subvecs.f4 [768.0 KB]


Lets have a look at what we downloaded - you'll see its very small

In [6]:
!ls -d {index.path}
!ls -lh {index.path}

/root/.pyterrier/artifacts/82a8bae57b6ae9f700d3d3f606d8e64117a1cd9b8849a15b46e7211d5eee86e3
total 1.3G
-rw-r--r-- 1 root root 810M Jul 15 04:43 codes.f4
-rw-r--r-- 1 root root  612 Jul 15 04:43 config.json
-rw-r--r-- 1 root root  206 Jul 15 04:43 docnos.npids
-rw-r--r-- 1 root root  180 Jul 15 04:43 file_inventory.csv
-rw-r--r-- 1 root root 418M Jul 15 04:44 model.safetensors
-rw-r--r-- 1 root root 2.3M Jul 15 04:44 opq.f4
-rw-r--r-- 1 root root  106 Jul 15 04:44 pt_meta.json
-rw-r--r-- 1 root root 768K Jul 15 04:44 subvecs.f4


You should see files as follows:

 - subvecs.f4: the centroid embeddings (768KB)
 - opq.f4: the OPQ rotation matrix (2.3MB)
 - codes.f4: the codebook assigning documents to centroids. (810MB)
 - config.json and model.safetensors: these are the updated weights of the TCT query encoder that was finetuned by JPQ (418MB))


The index loads the centroid embeddings, and the code assignments. The index is a factory object which provide a retriever method.

Quick check - does the index have 8.8M passages as expected?

In [7]:
len(index)

8841823

## Retrieval

Now lets start setting up. We need a TCT-ColBERT (HNP) query encoder. The following code downloads the TCT checkpoint from HuggingFace, and then updates the model to use the fine-tuned JPQ query encoder.



In [8]:
model = pyterrier_dr.TctColBert.hnp()
model.model = model.model.from_pretrained(index.path)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Formulate the retrieval pipeline and show the resulting schematic.

In [9]:
retr_pipe = model >> index.retriever_pq()
retr_pipe

(TctColBert('castorini/tct_colbert-v2-hnp-msmarco') >> JPQ-PQ)

Show a search on this pipeline.

This should have scores of 82.839 and 82.448 for the top 2 documents.

In [10]:
retr_pipe.search("what are chemical reactions?").head(2)

,qid,docid,docno,score,rank,query
0,1,1525431,1525431,82.839813,1,what are chemical reactions?
1,1,8572191,8572191,82.448662,2,what are chemical reactions?


Now verify the effectiveness - we'll use TREC 2019 Deep Learning track topics & qrels, and calculate nDCG@10.



In [11]:
from pyterrier.measures import nDCG
pt.Experiment(
    [retr_pipe],
    pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged").get_topics(),
    pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged").get_qrels(),
    [nDCG@10],
    validate='ignore'
)

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [7.43MB/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz: [00:00] [4.28kB] [25.0MB/s]


,name,nDCG@10
0,(TctColBert('castorini/tct_colbert-v2-hnp-msma...,0.669942


You should see a performance of 0.6699, which matches to 0.670 reported in the 2nd last row of Table 2 in the submission.